
# Representative Volume Study for PDMS Networks

When predicting the equilibrium shear modulus $G$ of polymer networks via
Monte Carlo network generation (:class:`~pylimer_tools_cpp.MCUniverseGenerator`),
on-demand entanglement sampling (:meth:`~pylimer_tools_cpp.MCUniverseGenerator.sample_entanglements`),
and :class:`~pylimer_tools_cpp.MEHPForceBalance2`, the simulation box must be large
enough to act as a **Representative Volume Element (RVE)**—avoiding both
finite-size bias (e.g. periodic self-looping or insufficient topological
sampling) and excessive sample-to-sample variance.

Here we study realistic Gaussian bead-spring models of polydimethylsiloxane
(PDMS, $\rho_{\text{bead}} \approx 2.23\,\text{nm}^{-3}$,
$\rho_{\text{ent}} \approx 0.0608\,\text{nm}^{-3}$,
$\langle b^2 \rangle \approx 1.11\,\text{nm}^2$) across cubic box sizes
$L \in [14, 110]\,\text{nm}$ (from $\sim 6\times 10^3$ up to
$\sim 3\times 10^6$ beads), varying:

1. **Chain length** $N$,
2. **Cross-link conversion** $p$,
3. **Stoichiometric imbalance** $r = f N_{\text{xlinks}} / (2 N_{\text{strands}})$, and
4. **Number of independent samples** $n_{\text{samples}}$ needed for a target precision.


In [ ]:
from pathlib import Path
import sys

if "__file__" in globals():
    _examples_dir = str(Path(__file__).resolve().parents[1])
    _fb_dir = str(Path(__file__).resolve().parent)
    for _p in (_examples_dir, _fb_dir):
        if _p not in sys.path:
            sys.path.insert(0, _p)

from _example_cache import get_example_cache_path, load_or_compute_cache  # noqa: E402
from _plotting import (  # noqa: E402
    plot_rve_chain_length_study,
    plot_rve_conversion_study,
    plot_rve_sample_count_study,
    plot_rve_stoichiometry_study,
)
from _study_runners import run_representative_volume_pdms_study  # noqa: E402
from pylimer_tools.io.bead_spring_parameter_provider import (  # noqa: E402
    ParameterType,
    Parameters,
    get_parameters_for_polymer,
)
import pylimer_tools_cpp as pt  # noqa: E402

## Single PDMS Realization with On-Demand Entanglement Sampling
Using :func:`~pylimer_tools.io.bead_spring_parameter_provider.get_parameters_for_polymer`
with ``ParameterType.GAUSSIAN``, we parameterize a realistic PDMS box of side
length $L = 25\,\text{nm}$, sample entanglements on demand, and evaluate
$G$ with :class:`~pylimer_tools_cpp.MEHPForceBalance2`.



In [ ]:
params = get_parameters_for_polymer("PDMS", parameter_type=ParameterType.GAUSSIAN)
assert isinstance(params, Parameters)
dist_unit = params.get("distance_units")
b = params.get("<b>").to(dist_unit).magnitude
b2 = params.get("<b^2>").to(dist_unit**2).magnitude
r02 = params.get("R02").to(dist_unit**2).magnitude
gamma_conv = params.get_gamma_conversion_factor().to("MPa").magnitude

box_l = 25.0
vol = box_l**3
strand_len, p_demo, r_demo = 100, 0.95, 1.0
n_strands = int(round(params.get_bead_density() * vol / (strand_len + 0.5 * r_demo)))
n_xlinks = int(round(0.5 * r_demo * n_strands))

gen = pt.MCUniverseGenerator(box_l, box_l, box_l)
gen.set_seed(42)
gen.config_nr_of_mc_steps(0)
gen.set_bead_distance(b)
gen.set_mean_squared_bead_distance(b2)
gen.add_strands(n_strands, [strand_len] * n_strands)
gen.add_crosslinkers(n_xlinks, 4)
gen.link_strands_to_conversion(p_demo)

es = gen.sample_entanglements(
    nr_of_entanglements_to_sample=int(params.get_entanglement_density() * vol),
    upper_cutoff=params.get_sampling_cutoff(),
    seed="demo_rve",
    filtering_mode=pt.EntanglementFilteringMode.OMIT_INACTIVE,
)
fb = gen.get_force_balance2(es, entanglements_as_springs=False)
fb.run_force_relaxation(pt.StructureSimplificationMode.INACTIVE_THEN_X2F)
g_demo = gamma_conv * fb.get_gamma_factor(r02, nr_of_chains=1) / vol
print(
    f"Demo PDMS box (L={box_l:.0f} nm, N_strands={n_strands}): "
    f"G = {g_demo:.4f} MPa, w_active = {fb.get_active_weight_fraction():.3f}"
)

cache_path = get_example_cache_path(
    "representative_volume_pdms_cache.npz", "force_balance"
)
data = load_or_compute_cache(cache_path, run_representative_volume_pdms_study)

## Effect of Chain Length ($N$)
Varying the precursor strand length $N \in \{50, 100, 200, 400\}$ at
fixed $p = 0.95$ and $r = 1.0$ shows that longer chains require
significantly larger physical box sizes $L$ to converge. Because the
unperturbed end-to-end distance scales as $R_0 = \sqrt{N \langle b^2 \rangle}$
(reaching $\sim 21\,\text{nm}$ at $N = 400$) and the cross-link
density drops as $1/N$, small boxes ($L \lesssim R_0$) overestimate
$G$ and exhibit large sample-to-sample variance. Normalizing by
$L / R_0$ shows that a box size of $L \gtrsim 3\text{--}4\,R_0$
reduces the coefficient of variation (CV) below 2–5%.



In [ ]:
plot_rve_chain_length_study(data)

## Effect of Cross-link Conversion ($p$)
Varying the cross-link conversion $p \in \{0.75, 0.85, 0.925, 0.98\}$ at
$N = 100$ and $r = 1.0$ approaches the classical gelation threshold
$p_{\text{gel}} = 1 / \sqrt{r(f - 1)} \approx 0.577$ from above. Near
$p = 0.75$, the active weight fraction drops sharply and spatial
connectivity fluctuations grow, inflating both the finite-size bias in small
boxes and the sample-to-sample CV by roughly an order of magnitude compared
to well-cross-linked networks ($p \ge 0.925$).



In [ ]:
plot_rve_conversion_study(data)

## Effect of Stoichiometric Imbalance ($r$)
Varying the stoichiometric ratio $r = f N_{\text{xlinks}} / (2 N_{\text{strands}})$
across $r \in \{0.65, 0.80, 1.00, 1.25\}$ (each reacted to 95% of its
stoichiometric maximum conversion, $p = 0.95 \min(1, 1/r)$) demonstrates
a similar broadening of the required RVE under cross-linker deficiency
($r = 0.65$), where sparse percolation paths increase sample-to-sample
scatter. By contrast, excess cross-linker ($r = 1.25$, corresponding to
95% strand conversion) preserves a high active fraction and converges nearly as
fast as the stoichiometric reference ($r = 1.00$).



In [ ]:
plot_rve_stoichiometry_study(data)

## Number of Samples vs. Representative Volume
For the reference PDMS network ($N = 100$, $p = 0.925$, $r = 1.0$),
we evaluate $n_{\text{samples}} = 24$ independent realizations at each box
size. While the 95% confidence interval half-width decreases as
$1/\sqrt{n_{\text{samples}}}$, averaging many small boxes ($L \le 20\,\text{nm}$)
cannot eliminate systematic finite-size bias. Choosing $L \ge 56\,\text{nm}$
($\sim 4\times 10^5$ beads) with $n_{\text{samples}} \approx 8\text{--}12$
independent realizations keeps both finite-size bias and statistical
uncertainty below 1–2%.



In [ ]:
plot_rve_sample_count_study(data)